In [47]:
# Import packages
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from transformers import DistilBertTokenizer, TFDistilBertForSequenceClassification
import tensorflow as tf

In [25]:
# Load in DataFrame
url = './Data/Raw/WELFake_Dataset.csv'
df = pd.read_csv(url, encoding = 'latin1')
df.head()
df.drop(columns=["Unnamed: 0"])

# Sample an even number of rows for each label
sample_size_per_label = 5000  

# Separate the DataFrame into two based on the labels
df_label_0 = df[df['label'] == 0]
df_label_1 = df[df['label'] == 1]

# Sample from each of these DataFrames
df_label_0_sample = df_label_0.sample(n=sample_size_per_label, random_state=1)
df_label_1_sample = df_label_1.sample(n=sample_size_per_label, random_state=1)

# Concatenate the samples back into a single DataFrame
df_sampled = pd.concat([df_label_0_sample, df_label_1_sample])

# Shuffle the sampled DataFrame to mix the rows from the two labels
df_sampled = df_sampled.sample(frac=1, random_state=1).reset_index(drop=True)



In [26]:
# Test/train/validation split
train_data, val_data = train_test_split(df_sampled, test_size=0.2, random_state=42)

Y_train = pd.DataFrame(train_data['label'])
X_train = train_data[['text', 'title']]
Y_val = pd.DataFrame(val_data['label'])
X_val = val_data[['text', 'title']]


print("Train data shape:", train_data.shape)
print("Validation_X data shape:", X_val.shape)
print("Validation_Y data shape:", Y_val.shape)

Train data shape: (8000, 4)
Validation_X data shape: (2000, 2)
Validation_Y data shape: (2000, 1)


In [27]:
# Load pre-trained DistilBERT model and tokenizer
disbert_tr = 'distilbert-base-uncased'
tokenizer = DistilBertTokenizer.from_pretrained(disbert_tr)
model = TFDistilBertForSequenceClassification.from_pretrained(disbert_tr)

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFDistilBertForSequenceClassification: ['vocab_transform.bias', 'vocab_layer_norm.bias', 'vocab_layer_norm.weight', 'vocab_transform.weight', 'vocab_projector.bias']
- This IS expected if you are initializing TFDistilBertForSequenceClassification from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFDistilBertForSequenceClassification from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
Some weights or buffers of the TF 2.0 model TFDistilBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['pre_classifier.weight', 'pre_classifier.bias', 'classifier.weight', 'classifier.bias']
You should 

In [28]:
max_length = 128

def tokenize_headlines(data):
    # Ensure all entries are converted to strings
    text_data = data.astype(str)
    
    return tokenizer(
        text_data.tolist(),  # Convert the text data into a list of strings directly
        truncation=True,
        padding='max_length',
        max_length=max_length,
        return_tensors='tf'
    )

# Assuming X_train and X_val DataFrames have a 'text' column
train_tokenized = tokenize_headlines(X_train["text"])
val_tokenized = tokenize_headlines(X_val["text"])


KeyboardInterrupt: 

In [ ]:
train_labels = Y_train['label'].tolist()
val_labels = Y_val['label'].tolist()

In [ ]:
# Prepare TensorFlow datasets
train_dataset = tf.data.Dataset.from_tensor_slices((dict(train_tokenized), train_labels)).shuffle(len(X_train)).batch(32)
val_dataset = tf.data.Dataset.from_tensor_slices((dict(val_tokenized), val_labels)).batch(32)

In [ ]:
# Compile and fit the model
optimizer = tf.keras.optimizers.Adam(learning_rate=0.005)
loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
metric = tf.keras.metrics.SparseCategoricalAccuracy('accuracy')

model.compile(optimizer=optimizer, loss=loss, metrics=[metric])
model.fit(train_dataset, epochs=1)

# Save the model
model.save("distilbert_fine_tuned_model")

247/247 [==============================] - 2415s 10s/step - loss: 0.0051 - accuracy: 0.9968


INFO:tensorflow:Assets written to: distilbert_fine_tuned_model\assets


INFO:tensorflow:Assets written to: distilbert_fine_tuned_model\assets


In [ ]:
loss, accuracy = model.evaluate(val_dataset)

print("Validation Loss:", loss)
print("Validation Accuracy:", accuracy)

31/31 [==============================] - 120s 4s/step - loss: 0.0000e+00 - accuracy: 1.0000
Validation Loss: 0.0
Validation Accuracy: 1.0


In [9]:
# Evaluate the model on the test set
loss, accuracy = model.evaluate(test_dataset)
print(f'Test Loss: {loss}, Test Accuracy: {accuracy}')

16/16 [==============================] - 66s 4s/step - loss: 0.0000e+00 - accuracy: 1.0000
Test Loss: 0.0, Test Accuracy: 1.0
